# Platonic Universe: Legacy Survey MKNN Embedding Alignment

This notebook loads the precomputed embeddings for the **Legacy Survey-HSC crossmatch** dataset (~100,000 galaxies) and computes the intramodal MKNN scores to replicate Table 1 from the paper.

Since the dataset is 14.69 GB, Colab is recommended as it has high network bandwidth.

In [ ]:
# Install Hugging Face datasets library in Colab environment
!pip install datasets

In [ ]:
from datasets import load_dataset
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

In [ ]:
# Pre-computed DINOv2 + ConvNeXt + ViT + AstroPT embeddings on LegacySurvey/HSC crossmatch
print("Loading dataset (14.69 GB)... This might take 1-2 minutes on Colab...")
ds = load_dataset("UniverseTBD/legacysurvey_hsc_embeddings", split="train")



In [ ]:
prefixes = ["astropt", "convnext", "dino", "ijepa", "vit"]
suffixes = ["hsc", "legacysurvey"]

In [ ]:
# Check features and row count
print(f"Total cleaned samples: {len(ds)}")
ds

In [ ]:
def MKNN(d1, d2, k):
    # Filter out rows where either embedding is None
    valid_idx = [i for i, (x, y) in enumerate(zip(d1, d2)) if x is not None and y is not None]
    if not valid_idx:
        return 0.0
    
    d1_clean = np.array([d1[i] for i in valid_idx])
    d2_clean = np.array([d2[i] for i in valid_idx])
    
    # Truncate outliers above 95th percentile
    threshold1 = np.percentile(np.abs(d1_clean), 95)
    d1_clean = np.clip(d1_clean, -threshold1, threshold1)
    
    threshold2 = np.percentile(np.abs(d2_clean), 95)
    d2_clean = np.clip(d2_clean, -threshold2, threshold2)
    
    # Find all k nearest neighbors (using cosine distance)
    knn_1 = NearestNeighbors(n_neighbors=k, metric=\cosine\).fit(d1_clean)
    knn_2 = NearestNeighbors(n_neighbors=k, metric=\cosine\).fit(d2_clean)
    knn_1 = knn_1.kneighbors(return_distance=False)
    knn_2 = knn_2.kneighbors(return_distance=False)

    matches = []
    for i in range(knn_1.shape[0]):
        matches.append(len(np.intersect1d(knn_1[i], knn_2[i])))
    return np.sum(matches) / (float(knn_1.shape[0]) * k)


In [ ]:
names = list(ds.features.keys())[1:]
names_legacy = [name for name in names if name.endswith("legacysurvey")]
names_hsc = [name for name in names if name.endswith("hsc")]

In [ ]:
# Compute MKNN for HSC pairs
hsc_pairs = {}
for i in range(1, len(names_hsc)):
    if names_hsc[i].split('_')[0] == names_hsc[i-1].split('_')[0]:
        hsc_pairs[f'{names_hsc[i-1]} vs {names_hsc[i]}'] = MKNN(ds[names_hsc[i-1]], ds[names_hsc[i]], 10)
        print(f'Done: {names_hsc[i-1]} vs {names_hsc[i]}')
print('\nHSC Pairs MKNN scores:')
print(hsc_pairs)


In [ ]:
# Compute MKNN for Legacy Survey pairs
legacy_pairs = {}
for i in range(1, len(names_legacy)):
    if names_legacy[i].split('_')[0] == names_legacy[i-1].split('_')[0]:
        legacy_pairs[f'{names_legacy[i-1]} vs {names_legacy[i]}'] = MKNN(ds[names_legacy[i-1]], ds[names_legacy[i]], 10)
        print(f'Done: {names_legacy[i-1]} vs {names_legacy[i]}')
print('\nLegacy Survey Pairs MKNN scores:')
print(legacy_pairs)


In [ ]:
# Assemble results into a comparative DataFrame
data = {}
for k1, v1 in hsc_pairs.items():
    model_key = k1.replace('_hsc', '')
    data[model_key] = {'hsc': v1 * 100, 'legacysurvey': None}

for k2, v2 in legacy_pairs.items():
    model_key = k2.replace('_legacysurvey', '')
    if model_key in data:
        data[model_key]['legacysurvey'] = v2 * 100
    else:
        data[model_key] = {'hsc': None, 'legacysurvey': v2 * 100}

df = pd.DataFrame.from_dict(data, orient='index')
print('\nFinal Comparison Table (MKNN %):')
print(df.to_string())
